# SCI1070 — Semaine 14 : préférences, substituts, et ce qu'une perte mesure vraiment

| Volet | Ce qu'il isole | Entraînement |
| --- | --- | --- |
| **A** | les deux formes de la perte de préférence | **aucun** |
| **B** | une chaîne de préférences complète | contrôle classique |
| **C** | la divergence entre substitut et récompense réelle | deux ou trois tailles |
| **D** | le surapprentissage d'un modèle de récompense | régression logistique |

**La chaîne de la première moitié de la semaine n'est pas reproductible ici** : treize
heures de processeur graphique, douze gigaoctets de segments, et **trois cents
jugements humains**. La quatrième contrainte exclut **toute** machine.

> **Aucun des quatre volets n'a été exécuté par ce cours.** Si votre résultat diffère,
> **dites ce que vous avez obtenu et expliquez ce que cela mesure.**

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle
from sci1070.graines import fixer_graines

device = torch.device("cpu")
L_SEGMENT = 50        # un segment compte 50 pas : c'est l'échelle du décalage

---
# Volet A : les deux pertes, comme fonctions pures

**C'est le meilleur volet de la semaine.** Aucun entraînement, vingt lignes, et le
point le plus tranchant.

In [ ]:
def perte_enoncee(R1, R2, y1, y2):
    """Entropie croisée binaire sur le LOGIT UNIQUE R1 - R2, contre la cible y1."""
    ...        # À COMPLÉTER : un seul logit, R1 - R2, et la cible y1


def perte_calculee(R1, R2, y1, y2):
    """Entropie croisée binaire sur les deux sommes EMPILÉES, contre l'étiquette complète."""
    ...        # À COMPLÉTER : les deux sommes empilées, et l'étiquette complète

### Étape 1 — l'invariance par décalage, et c'est la mesure centrale du volet

**Pourquoi cinquante et non dix** : un segment compte \(L = 50\) pas, donc ajouter un
à la récompense **par pas** ajoute cinquante à la somme.

In [ ]:
R1, R2 = torch.tensor(3.0), torch.tensor(1.0)
y1, y2 = torch.tensor(1.0), torch.tensor(0.0)
decalage = float(L_SEGMENT)        # +1 par pas, sur 50 pas

avant = (perte_enoncee(R1, R2, y1, y2), perte_calculee(R1, R2, y1, y2))
apres = (perte_enoncee(R1 + decalage, R2 + decalage, y1, y2),
         perte_calculee(R1 + decalage, R2 + decalage, y1, y2))

print(f"forme énoncée  : {float(avant[0]):.6f} -> {float(apres[0]):.6f}")
print(f"forme calculée : {float(avant[1]):.6f} -> {float(apres[1]):.6f}")
assert torch.isclose(avant[0], apres[0]), "la forme énoncée ne doit PAS bouger"
assert not torch.isclose(avant[1], apres[1]), "la forme calculée DOIT bouger"

### Étape 2 — la paire « les deux sont bons »

In [ ]:
grille = torch.linspace(-10, 10, 121)
A, B = torch.meshgrid(grille, grille, indexing="ij")
demi = torch.tensor(0.5)

cartes = {}
for nom, perte in [("forme énoncée", perte_enoncee), ("forme calculée", perte_calculee)]:
    carte = ...        # À COMPLÉTER : un tableau pour la grille
    ...        # À COMPLÉTER : évaluer la perte sur toute la grille
    cartes[nom] = carte

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nom, carte) in zip(axes, cartes.items()):
    im = ax.pcolormesh(grille, grille, carte.T, shading="auto")
    ax.set_title(nom); ax.set_xlabel("$R_1$"); ax.set_ylabel("$R_2$")
    ax.plot([-10, 10], [-10, 10], "w--", linewidth=1)
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

print("Pour la forme énoncée : une VALLÉE le long de la diagonale R1 = R2.")
print("Pour la forme calculée : un UNIQUE minimum à l'origine.")
print()
print("Ce que la seconde apprend au modèle sur deux segments que la personne")
print("trouvait tous les deux excellents : qu'ils valent ZÉRO, et non qu'ils")
print("se valent. Elle transforme un jugement RELATIF en cible ABSOLUE.")

### Étape 3 — les cibles absolues contradictoires

In [ ]:
...        # À COMPLÉTER : trois segments, deux paires, et le gradient par rapport au MOYEN

Pour la forme **énoncée**, les deux gradients pointent en sens contraires, et c'est
**correct** : le segment moyen doit monter par rapport au mauvais et descendre par
rapport au bon. Pour la forme **calculée**, chaque paire impose au moyen une cible
**absolue** (1 dans un cas, 0 dans l'autre) : le modèle reçoit deux ordres
incompatibles pour le même segment.

### Étape 4 — la dérive d'échelle

In [ ]:
...        # À COMPLÉTER : multiplier les deux sommes par 2, 5, 10, et tracer

**Ce que ce volet établit** : que les deux objectifs **ne dépendent pas des mêmes
quantités**. **Ce qu'il n'établit pas** : que l'un des deux produit un meilleur
agent. **Dites-le dans votre conclusion.**

---
# Volet B : une chaîne de préférences complète, à un millième du coût

**Une seule pièce est remplacée, et c'est la bonne : la personne est remplacée par
une règle que vous écrivez vous-même.**

In [ ]:
class EnregistreurSegments(gym.Wrapper):
    def __init__(self, env, longueur=25, probabilite=0.02, graine=0):
        super().__init__(env)
        ...        # À COMPLÉTER : le tampon courant, la liste des segments, et le compteur

    def step(self, action):
        obs, recompense, termine, tronque, info = self.env.step(action)
        ...        # À COMPLÉTER : déclencher, accumuler, et fermer un segment à `longueur` pas
        return obs, recompense, termine, tronque, info

**Étape 2 — l'étiqueteur scripté.** Deux contraintes : **pas** la récompense de
l'environnement, et **exprimable en une phrase de français**.

In [ ]:
# Mon critère, en une phrase : je préfère le segment où le chariot reste en
# moyenne le plus près du centre du rail, sans tenir aucun compte de l'angle.
def etiqueter(segment_1, segment_2, tolerance=0.05):
    """Rend (y1, y2). Pense au cas (0,5 ; 0,5) : les deux sont bons."""
    ...        # À COMPLÉTER : le critère, et le cas d'égalité

In [ ]:
...        # À COMPLÉTER : étape 1 et 3 : collecter des segments, étiqueter, et ajuster le modèle

**Étape 4 — la politique contre le modèle**, avec la **normalisation glissante sur
cent valeurs**.

In [ ]:
class SubstitutRecompense(gym.Wrapper):
    """La récompense de l'environnement est REMPLACÉE par celle du modèle."""

    def __init__(self, env, modele, fenetre=100):
        super().__init__(env)
        self.modele = modele
        ...        # À COMPLÉTER : la normalisation glissante sur cent valeurs

    def step(self, action):
        obs, recompense_env, termine, tronque, info = self.env.step(action)
        ...        # À COMPLÉTER : la récompense du modèle, normalisée
        info["recompense_environnement"] = float(recompense_env)   # journalisée
        return obs, recompense_modele, termine, tronque, info

**Étape 5, la variante semi-supervisée.** La référence qui **ignore** les épisodes
non étiquetés apprend nettement plus lentement, et **dire pourquoi est la question
de l'étape** : elle jette neuf dixièmes de son expérience, donc à budget de pas égal
elle dispose de dix fois moins de données.

---
# Volet C : faire diverger le substitut et la récompense réelle

In [ ]:
# Type construit (section 14.12) : une ERREUR DE PORTÉE. Le substitut mesure le
# même désidérata (rester en équilibre) mais seulement sur la moitié droite du
# rail, et ignore entièrement ce qui se passe à gauche.
def recompense_reelle(obs, action, info):
    return 1.0                       # l'intention : tenir le plus longtemps possible


def recompense_substitut(obs, action, info):
    ...        # À COMPLÉTER : le MÊME désidérata, mesuré sur une partie seulement de l'espace

In [ ]:
...        # À COMPLÉTER : entraîner sur le SUBSTITUT seul, avec deux ou trois capacités

> **Je vous préviens tout de suite, parce que c'est le point où l'honnêteté du volet
> se joue : il est très possible que vous n'obteniez aucune séparation.** Les
> transitions de la section 14.13 apparaissent sur des plages de capacité de **deux à
> quatre ordres de grandeur**, et vous en couvrez **un**. **Si vous ne voyez rien,
> c'est un résultat**, et il se rapporte ainsi, avec l'écart d'échelle comme
> explication candidate.

**Étape 4 — la corrélation, et c'est l'étape que je préfère.**

In [ ]:
...        # À COMPLÉTER : la corrélation de Pearson aux DEUX points de contrôle

---
# Volet D : le surapprentissage d'un modèle de récompense

In [ ]:
...        # À COMPLÉTER : trois tailles d'étiquetage, arrêt anticipé sur la partition réservée

In [ ]:
...        # À COMPLÉTER : étape 4, facultative : un étiqueteur qui se trompe une fois sur dix

**Ce que ce volet établit** : l'ordre de grandeur du régime de données dans lequel un
modèle de récompense cesse de surapprendre, **sur votre tâche**. **Ce qu'il n'établit
pas** : rien de transférable à une tâche visuelle, où le modèle de la section 14.6
avait **plus de cent mille dimensions d'entrée et trois cents exemples**.

---
# La solution courte, et pourquoi elle ne dispense pas du volet A

In [ ]:
# A executer APRES avoir ecrit la votre, jamais avant.
from stable_baselines3 import PPO

# L'enveloppe de substitution est la VOTRE : c'est elle qui porte la semaine.
env_substitut = SubstitutRecompense(gym.make("CartPole-v1"), modele)
modele_court = PPO("MlpPolicy", env_substitut, seed=0, verbose=0)
modele_court.learn(total_timesteps=20_000)
# Puis evaluer sur la recompense REELLE, pas sur le substitut.
print("entraîné contre le modèle de récompense ; à évaluer sur la vraie tâche")

> **En revanche, le volet A n'a aucune solution courte, et c'est délibéré : aucune
> bibliothèque ne vous dira de quoi une perte dépend.** Il faut l'évaluer soi-même,
> et cela prend cinq minutes.